# Local text-only VL baseline

Use the VL GGUF Lab kernel. Run one inference process at a time. This notebook calls the same implementation as the command line. The model closes when the run returns. Shut down the kernel and stop JupyterLab when finished. See docs/01-baseline.md for interpretation.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src/baseline.py').is_file(), 'Open from the project or notebooks folder'
sys.path.insert(0, str(ROOT / 'src'))
from baseline import run, check_memory
check_memory()

The next cell loads about 3.81 GB of model weights. Close other memory-heavy applications first. The vision projector is not loaded. Resume appends missing indexes only after checking dataset/model/settings provenance. Each attempt gets separate metadata; each row is flushed and synced to disk. Use max_new_rows=1 for a bounded probe, then remove that bound to finish the sample.

In [ ]:
output = run(output=ROOT / 'results/baseline.jsonl', limit=10, seed=42, resume=True, max_new_rows=1)
output

In [ ]:
import json
import statistics
rows = [json.loads(line) for line in output.read_text().splitlines()]
print('Completed:', len(rows))
print('Mean unigram F1:', statistics.mean(r['unigram_f1'] for r in rows))
print('Output caps reached:', sum(r['finish_reason'] == 'length' for r in rows))
print('Mean seconds per input:', statistics.mean(r['wall_seconds'] for r in rows))

Read raw outputs before judging quality. Unigram overlap and SHOT headings are weak diagnostics, not semantic scores. Shut down the notebook kernel and stop Jupyter when finished.